# Assignment 04 — Bài toán 2: Dự đoán Giá nhà Việt Nam bằng CNN 1D

**Môn học:** Intelligent System Development — TS. Trần Đình Quế
**Sinh viên:** Đinh Hải Triều — B23DCCN843 — Lớp 06

---

## Mục tiêu

1. Cài đặt **CNN 1D cho bài toán HỒI QUY** bằng NumPy from scratch, với
   **head tuyến tính** và hàm mất mát **MSE** — khác hẳn head Sigmoid của Bài 1.
2. Dựng hai bản tương đương bằng **PyTorch** và **TensorFlow/Keras**.
3. Đối sánh với **MLP 5 tầng (Assignment 03)** và **Gradient Boosting**.
4. Phân tích một vấn đề mô hình hoá **đặc thù của bài này**: sau one-hot, các
   cột liền kề là những **chỉ báo loại trừ nhau của cùng một biến**, nên cửa sổ
   tích chập trượt qua chúng mang ý nghĩa rất đáng ngờ.
5. Xuất `model_cnn.json` và gọi từ frontend để chạy serverless trên Vercel.

**Bài toán:** Regression — dự đoán `Price` (tỉ VNĐ) từ tỉnh/thành, loại hình,
diện tích, số phòng ngủ, phòng tắm và số tầng.

> **Nguyên lý "Kiến trúc theo sát Bài toán" (slide 27–28).** Với hồi quy, nơ-ron
> đầu ra phải **tuyến tính**, không Sigmoid — vì giá nhà không bị chặn trong
> $[0, 1]$. Đây là thay đổi bắt buộc, không phải tuỳ chọn.

In [1]:
# ============================================================================
# KHỐI 1 — Nạp thư viện, cố định seed và cấu hình hình vẽ
# ============================================================================
import json
import os
import pathlib
import time

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "font.family": "DejaVu Sans",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
sns.set_palette("deep")

ROOT = pathlib.Path.cwd()
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
print("Thư mục làm việc :", ROOT)
print("Thư mục hình vẽ  :", FIG)

Thư mục làm việc : C:\Users\admin\Downloads\bt-thay quế\tuan 1\website_dudoan_gia_nha
Thư mục hình vẽ  : C:\Users\admin\Downloads\bt-thay quế\tuan 1\website_dudoan_gia_nha\figures


## 1. Nạp dữ liệu và khảo sát ban đầu (EDA)

In [2]:
# ============================================================================
# KHỐI 2 — Nạp dữ liệu, xem phân bố giá và độ lệch phải
# ============================================================================
df = pd.read_csv(ROOT / "data" / "vietnam_housing_dataset.csv")
print("Kích thước:", df.shape)
print("\nKiểu dữ liệu:")
print(df.dtypes.to_string())
print("\nGiá trị thiếu:", int(df.isna().sum().sum()))

print("\nThống kê giá (tỉ VNĐ):")
print(df["Price"].describe().round(3).to_string())
print(f"\nHệ số bất đối xứng (skewness) của Price : {df['Price'].skew():.3f}")
print(f"Hệ số bất đối xứng của log1p(Price)     : {np.log1p(df['Price']).skew():.3f}")
print(f"Tỉ lệ giá lớn nhất / giá trung vị       : {df['Price'].max()/df['Price'].median():.1f} lần")
df.head()

Kích thước: (2000, 7)

Kiểu dữ liệu:
Province         str
Area         float64
Bedrooms       int64
Bathrooms      int64
Floors         int64
HouseType        str
Price        float64

Giá trị thiếu: 0

Thống kê giá (tỉ VNĐ):
count    2000.000
mean       20.126
std        17.675
min         1.617
25%         8.818
50%        14.764
75%        25.421
max       200.000

Hệ số bất đối xứng (skewness) của Price : 2.949
Hệ số bất đối xứng của log1p(Price)     : 0.279
Tỉ lệ giá lớn nhất / giá trung vị       : 13.5 lần


,Province,Area,Bedrooms,Bathrooms,Floors,HouseType,Price
0,Đồng Nai,29.3,1,1,3,Nhà phố,6.113
1,Hải Phòng,27.5,3,3,2,Đất nền,4.713
2,Khánh Hòa,168.3,4,4,1,Chung cư,50.060
3,Cần Thơ,65.9,4,5,1,Nhà phố,15.835
4,Đồng Nai,33.5,2,3,2,Nhà phố,8.961


In [3]:
# ============================================================================
# KHỐI 3 — Hình 1: bốn góc nhìn EDA cho bài hồi quy
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(13.5, 9))

axes[0, 0].hist(df["Price"], bins=60, color="#10b981", edgecolor="white")
axes[0, 0].axvline(df["Price"].median(), color="#111827", ls="--",
                   label=f"Trung vị = {df['Price'].median():.1f}")
axes[0, 0].axvline(df["Price"].mean(), color="#ef4444", ls="--",
                   label=f"Trung bình = {df['Price'].mean():.1f}")
axes[0, 0].set_title(f"(a) Phân bố Price — lệch phải mạnh (skew={df['Price'].skew():.2f})",
                     fontweight="bold")
axes[0, 0].set_xlabel("Giá (tỉ VNĐ)")
axes[0, 0].set_ylabel("Số bất động sản")
axes[0, 0].legend()

axes[0, 1].hist(np.log1p(df["Price"]), bins=60, color="#6366f1", edgecolor="white")
axes[0, 1].set_title(f"(b) log1p(Price) — gần chuẩn (skew={np.log1p(df['Price']).skew():.2f})",
                     fontweight="bold")
axes[0, 1].set_xlabel("log1p(Giá)")
axes[0, 1].set_ylabel("Số bất động sản")

order = df.groupby("HouseType")["Price"].median().sort_values().index
sns.boxplot(data=df, x="HouseType", y="Price", order=order, ax=axes[1, 0],
            palette="viridis", showfliers=False, hue="HouseType", legend=False)
axes[1, 0].set_yscale("log")
axes[1, 0].set_title("(c) Giá theo loại hình (thang log)", fontweight="bold")
axes[1, 0].set_xlabel("")
axes[1, 0].set_ylabel("Giá (tỉ VNĐ, log)")
axes[1, 0].tick_params(axis="x", rotation=20)

sc = axes[1, 1].scatter(df["Area"], df["Price"], c=df["Bedrooms"], cmap="plasma",
                        s=14, alpha=0.6)
axes[1, 1].set_xscale("log")
axes[1, 1].set_yscale("log")
axes[1, 1].set_title("(d) Diện tích vs Giá (log–log), màu = số phòng ngủ", fontweight="bold")
axes[1, 1].set_xlabel("Diện tích (m², log)")
axes[1, 1].set_ylabel("Giá (tỉ VNĐ, log)")
plt.colorbar(sc, ax=axes[1, 1], label="Bedrooms")

plt.tight_layout()
plt.savefig(FIG / "c2_fig1_eda.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_17640\3730285119.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Tiền xử lý

### 2.1. One-Hot Encoding thay vì Label Encoding

`Province` và `HouseType` là biến **định danh** — gán số 0..9 sẽ áp một thứ tự
giả lên mạng ("Hà Nội = 3 < TP.HCM = 7"). Ta dùng **one-hot**: mỗi hạng mục
thành một chiều riêng.

### 2.2. Học trên thang log

Giá nhà lệch phải rất mạnh. Nếu tối thiểu hoá MSE trên giá gốc, một biệt thự
300 tỉ đóng góp sai số bình phương gấp hàng nghìn lần một căn 2 tỉ — mạng sẽ
chỉ tối ưu cho nhóm siêu đắt. Học `log1p(Price)` biến sai số tuyệt đối thành
**sai số tương đối**, đúng với cách thị trường định giá.

In [4]:
# ============================================================================
# KHỐI 4 — One-hot hai biến định danh, ghép thành ma trận đặc trưng
# ============================================================================
CAT = ["Province", "HouseType"]
NUM = ["Area", "Bedrooms", "Bathrooms", "Floors"]

provinces = sorted(df["Province"].unique())
house_types = sorted(df["HouseType"].unique())
print(f"Province ({len(provinces)}): {provinces}")
print(f"HouseType ({len(house_types)}): {house_types}")

X_num = df[NUM].values.astype(float)
X_cat = np.hstack([
    np.eye(len(provinces))[[provinces.index(v) for v in df["Province"]]],
    np.eye(len(house_types))[[house_types.index(v) for v in df["HouseType"]]],
])
X_all = np.hstack([X_num, X_cat])
FEATURE_NAMES = (NUM + [f"Province={p}" for p in provinces]
                 + [f"HouseType={t}" for t in house_types])
y_all = df["Price"].values.astype(float).reshape(-1, 1)
N_FEAT = X_all.shape[1]

print(f"\nMa trận đặc trưng sau one-hot: {X_all.shape}  "
      f"({len(NUM)} số + {len(provinces)} tỉnh + {len(house_types)} loại hình)")

Province (10): ['Bà Rịa-VT', 'Bình Dương', 'Cần Thơ', 'Hà Nội', 'Hải Phòng', 'Khánh Hòa', 'Quảng Ninh', 'TP.HCM', 'Đà Nẵng', 'Đồng Nai']
HouseType (5): ['Biệt thự', 'Chung cư', 'Nhà phố', 'Nhà trọ', 'Đất nền']

Ma trận đặc trưng sau one-hot: (2000, 19)  (4 số + 10 tỉnh + 5 loại hình)


In [5]:
# ============================================================================
# KHỐI 5 — Chia dữ liệu, chuẩn hoá đặc trưng số, biến đổi nhãn sang log
# Scaler chỉ fit trên TRAIN; các cột one-hot giữ nguyên 0/1.
# ============================================================================
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X_all, y_all, test_size=0.15, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, random_state=SEED)

scaler = StandardScaler().fit(X_train[:, :len(NUM)])


def prep(X):
    out = X.copy()
    out[:, :len(NUM)] = scaler.transform(X[:, :len(NUM)])
    return out


Xtr_flat, Xva_flat, Xte_flat = prep(X_train), prep(X_val), prep(X_test)

ytr_log = np.log1p(y_train)
Y_MEAN, Y_STD = float(ytr_log.mean()), float(ytr_log.std())
ztr = (ytr_log - Y_MEAN) / Y_STD
zva = (np.log1p(y_val) - Y_MEAN) / Y_STD
zte = (np.log1p(y_test) - Y_MEAN) / Y_STD

print(f"Train {Xtr_flat.shape} | Val {Xva_flat.shape} | Test {Xte_flat.shape}")
print(f"log1p(Price) trên train: mean={Y_MEAN:.4f}, std={Y_STD:.4f}")

# Thêm trục kênh cho CNN: (N, 19) -> (N, 1, 19)
Xtr = Xtr_flat[:, None, :]
Xva = Xva_flat[:, None, :]
Xte = Xte_flat[:, None, :]
print(f"\nSau khi thêm trục kênh (channels-first): {Xtr.shape}  (N, C_in, L)")

Train (1399, 19) | Val (301, 19) | Test (300, 19)
log1p(Price) trên train: mean=2.7900, std=0.6870

Sau khi thêm trục kênh (channels-first): (1399, 1, 19)  (N, C_in, L)


## 3. Một vấn đề mô hình hoá đặc thù: cửa sổ tích chập trượt qua khối one-hot

Bài 1 chỉ có 8 cột số, nên cửa sổ tích chập ít ra còn trượt qua những đại lượng
**cùng loại**. Ở bài này, sau one-hot, chuỗi đầu vào dài 19 có cấu trúc:

```
vị trí:  0    1    2    3  |  4 ... 12        |  13 ... 18
         Area Bed  Bath Flo|  Province one-hot |  HouseType one-hot
         ─── 4 cột SỐ ────  ── 9 cột CHỈ BÁO ─  ── 6 cột CHỈ BÁO ─
```

Hệ quả: một kernel dài 3 đặt giữa khối `Province` sẽ nhìn thấy ba chỉ báo
**loại trừ nhau** — trong đó **nhiều nhất một cái bằng 1**, còn lại bằng 0.
Phép tích chập khi đó rút gọn thành "lấy trọng số của đúng tỉnh đang bật", tức
một **bảng tra cứu**, chứ không phải phát hiện mẫu cục bộ.

Tệ hơn: ba tỉnh nào bị gộp chung một cửa sổ hoàn toàn do **thứ tự bảng chữ cái**
quyết định. Không có lý do thực tế nào để "Đà Nẵng, Hà Nội, Hải Phòng" hợp thành
một nhóm cục bộ có ý nghĩa.

In [6]:
# ============================================================================
# KHỐI 6 — Hình 2: minh hoạ cấu trúc khối của chuỗi đầu vào
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))

block_id = np.zeros(N_FEAT)
block_id[:len(NUM)] = 0
block_id[len(NUM):len(NUM) + len(provinces)] = 1
block_id[len(NUM) + len(provinces):] = 2

sample_row = Xtr[0, 0]
bar_colors = ["#0ea5e9", "#f59e0b", "#a855f7"]
axes[0].bar(range(N_FEAT), sample_row,
            color=[bar_colors[int(b)] for b in block_id], width=0.72)
for start, width, name, col in [
        (0, len(NUM), "4 cột SỐ", "#0ea5e9"),
        (len(NUM), len(provinces), f"{len(provinces)} chỉ báo Province", "#f59e0b"),
        (len(NUM) + len(provinces), len(house_types),
         f"{len(house_types)} chỉ báo HouseType", "#a855f7")]:
    axes[0].axvspan(start - 0.5, start + width - 0.5, color=col, alpha=0.10)
    axes[0].text(start + width / 2 - 0.5, axes[0].get_ylim()[1] * 0.92, name,
                 ha="center", fontsize=8, fontweight="bold", color=col)
axes[0].set_xticks(range(N_FEAT))
axes[0].set_xticklabels([n[:13] for n in FEATURE_NAMES], rotation=90, fontsize=6.5)
axes[0].set_title("(a) Một bất động sản dưới dạng chuỗi 1D dài 19", fontweight="bold")
axes[0].set_ylabel("Giá trị sau chuẩn hoá")

# Với mỗi cửa sổ dài 3, đếm xem nó chứa bao nhiêu cột one-hot
K = 3
win_onehot = [int(np.sum(block_id[i:i + K] > 0)) for i in range(N_FEAT - K + 1)]
win_mixed = [1 if len(set(block_id[i:i + K])) > 1 else 0 for i in range(N_FEAT - K + 1)]
axes[1].bar(range(len(win_onehot)), win_onehot, color="#f59e0b", width=0.7,
            label="số cột chỉ báo trong cửa sổ")
for i, m in enumerate(win_mixed):
    if m:
        axes[1].plot(i, win_onehot[i] + 0.16, "v", color="#ef4444", markersize=7)
axes[1].set_xticks(range(len(win_onehot)))
axes[1].set_xlabel("Vị trí cửa sổ tích chập (K = 3)")
axes[1].set_ylabel("Số cột one-hot trong cửa sổ")
axes[1].set_ylim(0, 3.7)
axes[1].set_title("(b) ▼ đỏ = cửa sổ vắt qua ranh giới hai khối khác loại",
                  fontweight="bold", fontsize=9.5)
axes[1].legend(fontsize=8)

plt.suptitle("Hình 2 — Cấu trúc khối khiến 'cửa sổ cục bộ' mất ý nghĩa",
             fontweight="bold", y=1.04)
plt.tight_layout()
plt.savefig(FIG / "c2_fig2_blocks.png", bbox_inches="tight")
plt.show()

n_pure_onehot = sum(1 for w in win_onehot if w == K)
print(f"Tổng số cửa sổ dài {K}: {len(win_onehot)}")
print(f"  • Cửa sổ nằm TRỌN trong khối one-hot : {n_pure_onehot}"
      f"  -> tích chập rút gọn thành bảng tra cứu")
print(f"  • Cửa sổ vắt qua ranh giới hai khối  : {sum(win_mixed)}"
      f"  -> trộn đại lượng không cùng đơn vị")
print(f"  • Cửa sổ chỉ chứa cột số             : "
      f"{sum(1 for w in win_onehot if w == 0)}")

Tổng số cửa sổ dài 3: 17
  • Cửa sổ nằm TRỌN trong khối one-hot : 13  -> tích chập rút gọn thành bảng tra cứu
  • Cửa sổ vắt qua ranh giới hai khối  : 4  -> trộn đại lượng không cùng đơn vị
  • Cửa sổ chỉ chứa cột số             : 2


C:\Users\admin\AppData\Local\Temp\ipykernel_17640\909384426.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Cài đặt CNN 1D thuần NumPy

In [7]:
import numpy as np

# ============================================================================
# 1. HÀM KÍCH HOẠT
# ============================================================================


def relu(z):
    """f(z) = max(0, z)."""
    return np.maximum(0.0, z)


def relu_grad(z):
    """f'(z) = 1 nếu z > 0, ngược lại 0. Tại z = 0 ta quy ước đạo hàm bằng 0."""
    return (z > 0).astype(z.dtype)


def sigmoid(z):
    """Ổn định số học: tách nhánh z >= 0 và z < 0 để exp() không tràn số."""
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out


def softmax(z):
    """Trừ max theo hàng trước khi exp — kỹ thuật log-sum-exp chống tràn số."""
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


# ============================================================================
# 2. TÍCH CHẬP 1D Ở MỨC HÀM
# ============================================================================
#
# Định nghĩa toán học dùng trong báo cáo (tích chập 'valid', stride 1):
#
#     z[n, o, l] = sum_{c=0}^{C_in-1} sum_{k=0}^{K-1} W[o, c, k] * x[n, c, l+k] + b[o]
#
# Bản `conv1d_naive` viết đúng ba vòng lặp như công thức trên — dễ đọc, dùng để
# kiểm chứng. Bản `conv1d_forward` dùng thủ thuật im2col nên nhanh hơn hàng chục
# lần mà vẫn cho kết quả giống hệt (notebook có assert đối chiếu hai bản).


def conv1d_naive(X, W, b):
    """Tích chập theo đúng định nghĩa toán học — ba vòng lặp tường minh.

    Chậm, nhưng là bản "nguyên văn công thức" để đối chiếu với bản im2col.
    """
    N, C_in, L_in = X.shape
    C_out, C_in_w, K = W.shape
    assert C_in == C_in_w, f"Số kênh vào không khớp: X có {C_in}, W cần {C_in_w}"
    L_out = L_in - K + 1

    Z = np.zeros((N, C_out, L_out))
    for o in range(C_out):          # từng bộ lọc
        for l in range(L_out):      # từng vị trí cửa sổ trượt
            for k in range(K):      # từng ô trong kernel
                Z[:, o, l] += np.sum(W[o, :, k] * X[:, :, l + k], axis=1)
            Z[:, o, l] += b[o]
    return Z


def im2col_1d(X, K):
    """Dàn mọi cửa sổ trượt thành một ma trận để biến tích chập thành nhân ma trận.

    Trả về `cols` có shape (N, C_in * K, L_out) với quy ước chỉ số:

        cols[n, c * K + k, l] = X[n, c, l + k]

    Nhờ layout `c * K + k`, mảng W shape (C_out, C_in, K) chỉ cần `reshape`
    (không cần transpose) là khớp: W.reshape(C_out, C_in*K)[o, c*K+k] = W[o,c,k].
    """
    N, C_in, L_in = X.shape
    L_out = L_in - K + 1
    cols = np.empty((N, C_in * K, L_out), dtype=X.dtype)
    for k in range(K):
        # Bước nhảy K: chỉ số k, K+k, 2K+k, ... đúng là c*K+k với c = 0, 1, 2, ...
        cols[:, k::K, :] = X[:, :, k:k + L_out]
    return cols


def col2im_1d(dcols, x_shape, K):
    """Phép chuyển vị của im2col: cộng dồn gradient về đúng vị trí trong X.

    Một phần tử x[n, c, i] tham gia nhiều cửa sổ khác nhau, nên gradient của nó
    là TỔNG các đóng góp — đây chính là chỗ phép cộng dồn (`+=`) là bắt buộc.
    """
    N, C_in, L_in = x_shape
    L_out = L_in - K + 1
    dX = np.zeros(x_shape, dtype=float)
    for k in range(K):
        dX[:, :, k:k + L_out] += dcols[:, k::K, :]
    return dX


def conv1d_forward(X, W, b):
    """Tích chập 'valid' stride 1 bằng im2col. Trả về (Z, cache)."""
    C_out, C_in, K = W.shape
    cols = im2col_1d(X, K)                              # (N, C_in*K, L_out)
    Wr = W.reshape(C_out, C_in * K)                     # (C_out, C_in*K)
    Z = np.einsum("ok,nkl->nol", Wr, cols, optimize=True) + b[None, :, None]
    return Z, (cols, X.shape, W.shape)


def conv1d_backward(dZ, W, cache):
    """Lan truyền ngược qua tích chập. Trả về (dX, dW, db).

    Cho dZ = dL/dZ shape (N, C_out, L_out), ba công thức suy ra trực tiếp từ
    z = W·cols + b:

        dW[o, c, k] = sum_{n,l} dZ[n,o,l] * cols[n, c*K+k, l]
        db[o]       = sum_{n,l} dZ[n,o,l]
        dcols       = W^T · dZ   ->  dX = col2im(dcols)

    Chú ý db là tổng trên CẢ batch VÀ mọi vị trí l: một bias duy nhất được dùng
    lại ở mọi vị trí (chia sẻ trọng số), nên gradient của nó gom hết đóng góp.
    Điều tương tự xảy ra với dW — đây chính là dấu vết toán học của việc chia sẻ
    trọng số, và là điểm khác biệt cốt lõi so với tầng Dense.
    """
    cols, x_shape, w_shape = cache
    C_out, C_in, K = w_shape

    dW = np.einsum("nol,nkl->ok", dZ, cols, optimize=True).reshape(w_shape)
    db = dZ.sum(axis=(0, 2))

    Wr = W.reshape(C_out, C_in * K)
    dcols = np.einsum("ok,nol->nkl", Wr, dZ, optimize=True)
    dX = col2im_1d(dcols, x_shape, K)
    return dX, dW, db


# ============================================================================
# 3. CÁC LỚP TẦNG
# ============================================================================


class Layer:
    """Giao diện chung. `params()` trả về list [(tên, mảng_trọng_số, hàm_gán)]."""

    trainable = False

    def forward(self, X, training=False):
        raise NotImplementedError

    def backward(self, dOut):
        raise NotImplementedError

    def param_list(self):
        """Trả về list các mảng tham số (để Adam cập nhật tại chỗ)."""
        return []

    def grad_list(self):
        return []

    def describe(self, in_shape):
        """(mô tả, out_shape, số tham số) — dùng để in bảng đặc tả tensor."""
        return (type(self).__name__, in_shape, 0)

    def to_dict(self, decimals):
        return {"type": type(self).__name__.lower()}


class Conv1D(Layer):
    """Tầng tích chập 1D: (N, C_in, L) -> (N, C_out, L-K+1).

    Khởi tạo He (Kaiming) với fan_in = C_in * K, phù hợp với ReLU đứng sau.
    """

    trainable = True

    def __init__(self, c_in, c_out, k, rng):
        self.c_in, self.c_out, self.k = c_in, c_out, k
        fan_in = c_in * k
        self.W = rng.normal(0.0, np.sqrt(2.0 / fan_in), (c_out, c_in, k))
        self.b = np.zeros(c_out)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X, training=False):
        Z, self._cache = conv1d_forward(X, self.W, self.b)
        return Z

    def backward(self, dZ):
        dX, self.dW, self.db = conv1d_backward(dZ, self.W, self._cache)
        return dX

    def param_list(self):
        return [self.W, self.b]

    def grad_list(self):
        return [self.dW, self.db]

    def describe(self, in_shape):
        C, L = in_shape
        out = (self.c_out, L - self.k + 1)
        n = self.c_out * self.c_in * self.k + self.c_out
        return (f"Conv1D(C {self.c_in}→{self.c_out}, K={self.k})", out, n)

    def to_dict(self, decimals):
        return {
            "type": "conv1d",
            "c_in": self.c_in, "c_out": self.c_out, "k": self.k,
            "W": np.round(self.W, decimals).tolist(),
            "b": np.round(self.b, decimals).tolist(),
        }


class ReLU(Layer):
    def forward(self, X, training=False):
        self._z = X
        return relu(X)

    def backward(self, dOut):
        return dOut * relu_grad(self._z)

    def describe(self, in_shape):
        return ("ReLU", in_shape, 0)

    def to_dict(self, decimals):
        return {"type": "relu"}


class MaxPool1D(Layer):
    """Gom cụm cực đại theo cửa sổ không chồng lấn, size = stride = `p`.

    Nếu L không chia hết cho p, phần dư ở cuối bị bỏ (floor) — giống hành vi
    mặc định của nn.MaxPool1d và MaxPooling1D.
    """

    def __init__(self, p=2):
        self.p = p

    def forward(self, X, training=False):
        N, C, L = X.shape
        p = self.p
        L_out = L // p
        Xc = X[:, :, :L_out * p].reshape(N, C, L_out, p)
        self._argmax = Xc.argmax(axis=3)
        self._shape = X.shape
        return Xc.max(axis=3)

    def backward(self, dOut):
        N, C, L = self._shape
        p = self.p
        L_out = L // p
        dX = np.zeros((N, C, L_out * p))
        # Chỉ ô thắng (đạt cực đại) nhận gradient, các ô còn lại nhận 0.
        n_i, c_i, l_i = np.ogrid[:N, :C, :L_out]
        dXc = dX.reshape(N, C, L_out, p)
        dXc[n_i, c_i, l_i, self._argmax] = dOut
        full = np.zeros(self._shape)
        full[:, :, :L_out * p] = dXc.reshape(N, C, L_out * p)
        return full

    def describe(self, in_shape):
        C, L = in_shape
        return (f"MaxPool1D(p={self.p})", (C, L // self.p), 0)

    def to_dict(self, decimals):
        return {"type": "maxpool1d", "p": self.p}


class GlobalMaxPool1D(Layer):
    """Lấy giá trị lớn nhất trên toàn trục thời gian: (N, C, L) -> (N, C).

    Dùng cho văn bản: mỗi bộ lọc trả lời "mẫu cục bộ mà tôi phụ trách có xuất
    hiện ở đâu đó trong câu hay không", nên độ dài câu không còn ảnh hưởng tới
    số chiều đầu ra — đó là cách CNN xử lý câu dài ngắn khác nhau.
    """

    def forward(self, X, training=False):
        self._argmax = X.argmax(axis=2)
        self._shape = X.shape
        return X.max(axis=2)

    def backward(self, dOut):
        N, C, L = self._shape
        dX = np.zeros(self._shape)
        n_i, c_i = np.ogrid[:N, :C]
        dX[n_i, c_i, self._argmax] = dOut
        return dX

    def describe(self, in_shape):
        C, L = in_shape
        return ("GlobalMaxPool1D", (C,), 0)

    def to_dict(self, decimals):
        return {"type": "globalmaxpool1d"}


class Flatten(Layer):
    """(N, C, L) -> (N, C*L). Thứ tự dàn phẳng là C trước, L sau (C-order)."""

    def forward(self, X, training=False):
        self._shape = X.shape
        return X.reshape(X.shape[0], -1)

    def backward(self, dOut):
        return dOut.reshape(self._shape)

    def describe(self, in_shape):
        return ("Flatten", (int(np.prod(in_shape)),), 0)

    def to_dict(self, decimals):
        return {"type": "flatten"}


class Dense(Layer):
    """Tầng kết nối đầy đủ: (N, D) -> (N, M), W shape (D, M)."""

    trainable = True

    def __init__(self, d_in, d_out, rng):
        self.d_in, self.d_out = d_in, d_out
        self.W = rng.normal(0.0, np.sqrt(2.0 / d_in), (d_in, d_out))
        self.b = np.zeros(d_out)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X, training=False):
        self._x = X
        return X @ self.W + self.b

    def backward(self, dOut):
        self.dW = self._x.T @ dOut
        self.db = dOut.sum(axis=0)
        return dOut @ self.W.T

    def param_list(self):
        return [self.W, self.b]

    def grad_list(self):
        return [self.dW, self.db]

    def describe(self, in_shape):
        n = self.d_in * self.d_out + self.d_out
        return (f"Dense({self.d_in}→{self.d_out})", (self.d_out,), n)

    def to_dict(self, decimals):
        return {
            "type": "dense",
            "d_in": self.d_in, "d_out": self.d_out,
            "W": np.round(self.W, decimals).tolist(),
            "b": np.round(self.b, decimals).tolist(),
        }


class Dropout(Layer):
    """Inverted dropout — chia cho keep_prob ngay lúc train nên lúc suy luận
    không phải chỉnh gì, trọng số xuất ra JSON dùng trực tiếp được."""

    def __init__(self, rate, rng):
        self.rate = rate
        self.rng = rng

    def forward(self, X, training=False):
        if not training or self.rate <= 0.0:
            self._mask = None
            return X
        keep = 1.0 - self.rate
        self._mask = (self.rng.random(X.shape) < keep) / keep
        return X * self._mask

    def backward(self, dOut):
        return dOut if self._mask is None else dOut * self._mask

    def describe(self, in_shape):
        return (f"Dropout(p={self.rate})", in_shape, 0)

    def to_dict(self, decimals):
        return {"type": "dropout", "rate": self.rate}


class Embedding(Layer):
    """Tra bảng nhúng cho văn bản: (N, L) chỉ số nguyên -> (N, C_emb, L).

    Đầu ra đã ở dạng channels-first để Conv1D dùng trực tiếp. Chỉ số 0 được
    dành riêng cho token đệm (PAD) và vector của nó bị ghim bằng 0 (cả trong
    khởi tạo lẫn sau mỗi bước cập nhật), nên phần đệm không đóng góp gì vào
    tích chập.
    """

    trainable = True

    def __init__(self, vocab_size, dim, rng, pad_idx=0):
        self.vocab_size, self.dim, self.pad_idx = vocab_size, dim, pad_idx
        self.E = rng.normal(0.0, 0.05, (vocab_size, dim))
        self.E[pad_idx] = 0.0
        self.dE = np.zeros_like(self.E)

    def forward(self, idx, training=False):
        self._idx = idx.astype(np.int64)
        # (N, L, dim) -> (N, dim, L)
        return self.E[self._idx].transpose(0, 2, 1)

    def backward(self, dOut):
        # dOut: (N, dim, L) -> (N, L, dim), rồi cộng dồn theo chỉ số token.
        g = dOut.transpose(0, 2, 1)
        self.dE = np.zeros_like(self.E)
        # np.add.at cộng dồn đúng khi một token xuất hiện nhiều lần trong batch
        # (phép gán thường sẽ ghi đè và làm mất gradient).
        np.add.at(self.dE, self._idx.ravel(), g.reshape(-1, self.dim))
        self.dE[self.pad_idx] = 0.0
        return None  # Embedding là tầng đầu tiên — không cần truyền ngược nữa

    def param_list(self):
        return [self.E]

    def grad_list(self):
        return [self.dE]

    def describe(self, in_shape):
        (L,) = in_shape
        return (f"Embedding({self.vocab_size}→{self.dim})", (self.dim, L),
                self.vocab_size * self.dim)

    def to_dict(self, decimals):
        return {
            "type": "embedding",
            "vocab_size": self.vocab_size, "dim": self.dim, "pad_idx": self.pad_idx,
            "E": np.round(self.E, decimals).tolist(),
        }


# ============================================================================
# 4. BỘ CHỨA TUẦN TỰ + HUẤN LUYỆN
# ============================================================================


class CNN1D:
    """CNN 1D tuần tự, tự cài forward/backward/Adam.

    `task` quyết định đầu ra và hàm mất mát:
        binary      : 1 nơ-ron + Sigmoid  + Binary Cross-Entropy
        regression  : 1 nơ-ron + Linear   + Mean Squared Error
        multiclass  : C nơ-ron + Softmax  + Categorical Cross-Entropy

    Với cả ba, đạo hàm của loss theo pre-activation cuối rút gọn về (ŷ − y) —
    đó chính là lý do ta ghép Sigmoid/Softmax với Cross-Entropy và Linear
    với MSE, chi tiết ở Chương I của báo cáo.
    """

    def __init__(self, layers, task="binary", lr=1e-3, l2=0.0, seed=42,
                 class_weight=None, clip_norm=None):
        assert task in {"binary", "regression", "multiclass"}
        self.layers = layers
        self.task = task
        self.lr = lr
        self.l2 = l2
        self.clip_norm = clip_norm
        self.class_weight = None if class_weight is None else np.asarray(class_weight, float)
        self.rng = np.random.default_rng(seed)

        # Trạng thái Adam: một cặp (m, v) cho mỗi mảng tham số.
        self._params = [p for L in layers for p in L.param_list()]
        self.m = [np.zeros_like(p) for p in self._params]
        self.v = [np.zeros_like(p) for p in self._params]
        self.t = 0
        self.history = {"train_loss": [], "val_loss": [], "train_metric": [], "val_metric": []}

    # ----------------------------- forward ---------------------------------
    def forward(self, X, training=False):
        A = X
        for L in self.layers:
            A = L.forward(A, training=training)
        Z = A                                  # pre-activation của head
        if self.task == "binary":
            return sigmoid(Z), Z
        if self.task == "multiclass":
            return softmax(Z), Z
        return Z, Z

    # ------------------------------ loss -----------------------------------
    def _sample_w(self, y_true):
        if self.class_weight is None or self.task != "multiclass":
            return None
        return self.class_weight[y_true.argmax(1)].reshape(-1, 1)

    def loss(self, y_pred, y_true, logits=None):
        """Hàm mất mát. Nếu truyền `logits` (pre-activation của head) thì dùng dạng
        log-sum-exp / log-sigmoid.

        Vì sao phải quan tâm: cách viết "ngây thơ" `-log(p + eps)` gài một sai số
        hệ thống bằng eps/p. Khi mạng gán cho lớp đúng một xác suất rất nhỏ
        (p ~ 1e-8, hay gặp ở epoch đầu), sai số đó lên tới ~1e-4 tương đối và
        KHÔNG khớp với gradient giải tích (ŷ − y) — đủ để phép kiểm tra gradient
        bằng sai phân số thất bại dù backward hoàn toàn đúng.

        Dạng logit dưới đây không cần eps, ổn định số học, và nhất quán tuyệt đối
        với dZ = ŷ − y. Đây cũng chính là lý do PyTorch khuyên dùng
        `binary_cross_entropy_with_logits` thay cho `sigmoid` rồi `log`.
        """
        n = y_true.shape[0]
        eps = 1e-12
        if self.task == "binary":
            if logits is not None:
                z = logits
                # max(z,0) − z·y + log(1 + exp(−|z|)) — BCE-with-logits ổn định.
                base = float(np.mean(np.maximum(z, 0.0) - z * y_true
                                     + np.log1p(np.exp(-np.abs(z)))))
            else:
                base = -np.mean(y_true * np.log(y_pred + eps)
                                + (1 - y_true) * np.log(1 - y_pred + eps))
        elif self.task == "multiclass":
            if logits is not None:
                z = logits - logits.max(axis=1, keepdims=True)
                log_p = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
                per = -np.sum(y_true * log_p, axis=1, keepdims=True)
            else:
                per = -np.sum(y_true * np.log(y_pred + eps), axis=1, keepdims=True)
            w = self._sample_w(y_true)
            base = float(np.sum(per if w is None else per * w) / n)
        else:
            base = float(np.mean((y_pred - y_true) ** 2))
        reg = self.l2 * sum(np.sum(p * p) for p in self._params) / (2 * n) if self.l2 else 0.0
        return base + reg

    # ---------------------------- backward ---------------------------------
    def backward(self, y_pred, y_true):
        n = y_true.shape[0]
        dZ = (y_pred - y_true) / n
        if self.task == "regression":
            dZ = 2.0 * dZ
        w = self._sample_w(y_true)
        if w is not None:
            dZ = dZ * w

        d = dZ
        for L in reversed(self.layers):
            d = L.backward(d)
            if d is None:       # đã tới tầng Embedding
                break

        if self.l2:
            for L in self.layers:
                if L.trainable:
                    for p, g in zip(L.param_list(), L.grad_list()):
                        g += self.l2 * p / n

    # ------------------------------ Adam -----------------------------------
    def _adam(self, beta1=0.9, beta2=0.999, eps=1e-8):
        grads = [g for L in self.layers for g in L.grad_list()]

        if self.clip_norm:
            # Cắt chuẩn gradient toàn cục: giữ hướng, chỉ co độ dài. Bài văn bản
            # có gradient nhảy vọt khi một n-gram hiếm xuất hiện, nên cần thứ này.
            total = np.sqrt(sum(float(np.sum(g * g)) for g in grads))
            if total > self.clip_norm:
                scale = self.clip_norm / (total + 1e-12)
                grads = [g * scale for g in grads]

        self.t += 1
        for i, (p, g) in enumerate(zip(self._params, grads)):
            self.m[i] = beta1 * self.m[i] + (1 - beta1) * g
            self.v[i] = beta2 * self.v[i] + (1 - beta2) * (g * g)
            mhat = self.m[i] / (1 - beta1 ** self.t)
            vhat = self.v[i] / (1 - beta2 ** self.t)
            p -= self.lr * mhat / (np.sqrt(vhat) + eps)   # cập nhật TẠI CHỖ

        # Giữ vector PAD bằng 0 sau mỗi bước (Adam có thể đẩy nó lệch khỏi 0).
        for L in self.layers:
            if isinstance(L, Embedding):
                L.E[L.pad_idx] = 0.0

    # ------------------------------ metric ---------------------------------
    def _metric(self, X, y, batch=512):
        p = self.predict_proba(X, batch=batch)
        if self.task == "binary":
            return float(np.mean((p >= 0.5).astype(int) == y))
        if self.task == "multiclass":
            return float(np.mean(p.argmax(1) == y.argmax(1)))
        ss_res = float(np.sum((y - p) ** 2))
        ss_tot = float(np.sum((y - y.mean()) ** 2))
        return 1.0 - ss_res / ss_tot            # R^2

    # ------------------------------- fit -----------------------------------
    def fit(self, X, y, X_val=None, y_val=None, epochs=100, batch_size=32,
            verbose_every=10, patience=None, eval_batch=512, eval_subset=None,
            val_score_fn=None):
        """Chu trình 4 bước mỗi mini-batch: Forward → Loss → Backward → Update.

        `eval_subset`: nếu đặt, loss/metric TRAIN mỗi epoch chỉ tính trên một mẫu
        con cố định cỡ này (chọn một lần, không đổi giữa các epoch) thay vì toàn
        bộ tập train. Chỉ dùng cho bài văn bản 23k mẫu, nơi một lượt forward đầy
        đủ mỗi epoch đắt hơn cả việc huấn luyện. Early stopping vẫn dựa trên
        validation đầy đủ nên không ảnh hưởng tới việc chọn mô hình.

        `val_score_fn(y_true_onehot, y_prob) -> float`: nếu đặt, early stopping
        và việc giữ trọng số tốt nhất sẽ **cực đại hoá** điểm này thay vì cực
        tiểu hoá val_loss.

        Vì sao cần: với dữ liệu mất cân bằng nặng và cross-entropy CÓ TRỌNG SỐ
        LỚP, val_loss dao động mạnh và đạt cực tiểu rất sớm, trong khi Macro-F1
        vẫn còn đang lên. Chọn mô hình theo val_loss khi đó dừng quá sớm và cho
        mô hình kém hẳn. Tiêu chí chọn mô hình phải là chỉ số ta thật sự quan tâm.
        """
        n = X.shape[0]
        # Quy ước nội bộ: luôn CỰC TIỂU `best_val`. Khi dùng val_score_fn ta lấy
        # dấu âm của điểm số, nên một nhánh early-stopping duy nhất phục vụ cả hai.
        best_val, best_state, wait = np.inf, None, 0
        self.history.setdefault("val_score", [])

        if eval_subset is not None and eval_subset < n:
            sub = self.rng.choice(n, size=eval_subset, replace=False)
            X_tr_eval, y_tr_eval = X[sub], y[sub]
        else:
            X_tr_eval, y_tr_eval = X, y

        for ep in range(1, epochs + 1):
            idx = self.rng.permutation(n)
            for s in range(0, n, batch_size):
                sl = idx[s:s + batch_size]
                xb, yb = X[sl], y[sl]
                yp, _ = self.forward(xb, training=True)     # (1) Forward
                self.backward(yp, yb)                      # (3) Backward
                self._adam()                               # (4) Update

            tr_pred, tr_logit = self._eval(X_tr_eval, batch=eval_batch)
            tr_loss = self.loss(tr_pred, y_tr_eval, logits=tr_logit)   # (2) Loss
            self.history["train_loss"].append(tr_loss)
            self.history["train_metric"].append(self._metric(X_tr_eval, y_tr_eval, eval_batch))

            if X_val is not None:
                va_pred, va_logit = self._eval(X_val, batch=eval_batch)
                va_loss = self.loss(va_pred, y_val, logits=va_logit)
                self.history["val_loss"].append(va_loss)
                self.history["val_metric"].append(self._metric(X_val, y_val, eval_batch))

                if val_score_fn is not None:
                    score = float(val_score_fn(y_val, va_pred))
                    self.history["val_score"].append(score)
                    watched, label = -score, "val_score"
                else:
                    watched, label = va_loss, "val_loss"

                if patience is not None:
                    if watched < best_val - 1e-6:
                        best_val, wait = watched, 0
                        best_state = [p.copy() for p in self._params]
                    else:
                        wait += 1
                        if wait >= patience:
                            if verbose_every:
                                shown = -best_val if val_score_fn is not None else best_val
                                print(f"  ⏹ Early stopping tại epoch {ep} "
                                      f"({label} tốt nhất = {shown:.4f})")
                            break

            if verbose_every and (ep % verbose_every == 0 or ep == 1):
                msg = (f"  epoch {ep:4d} | train_loss={tr_loss:.4f} "
                       f"| train_metric={self.history['train_metric'][-1]:.4f}")
                if X_val is not None:
                    msg += (f" | val_loss={self.history['val_loss'][-1]:.4f}"
                            f" | val_metric={self.history['val_metric'][-1]:.4f}")
                    if val_score_fn is not None:
                        msg += f" | val_score={self.history['val_score'][-1]:.4f}"
                print(msg)

        if best_state is not None:
            # Trả tham số về trạng thái tốt nhất trên validation, ghi TẠI CHỖ để
            # self._params (và các layer) vẫn trỏ tới cùng mảng.
            for p, bp in zip(self._params, best_state):
                p[...] = bp
        return self

    # ---------------------------- inference --------------------------------
    def _eval(self, X, batch=512):
        """Suy luận theo lô, trả về (xác suất, logits). Logits cần cho `loss()`."""
        ps, zs = [], []
        for s in range(0, X.shape[0], batch):
            p, z = self.forward(X[s:s + batch], training=False)
            ps.append(p)
            zs.append(z)
        return np.concatenate(ps, axis=0), np.concatenate(zs, axis=0)

    def predict_proba(self, X, batch=512):
        return self._eval(X, batch)[0]

    def predict(self, X, threshold=0.5, batch=512):
        p = self.predict_proba(X, batch=batch)
        if self.task == "binary":
            return (p >= threshold).astype(int)
        if self.task == "multiclass":
            return p.argmax(1)
        return p

    # ------------------------- đặc tả và xuất JSON -------------------------
    def n_params(self):
        return int(sum(p.size for p in self._params))

    def shape_table(self, in_shape):
        """Bảng (tầng, đầu vào, đầu ra, số tham số) — phần đặc tả tensor của báo cáo."""
        rows, shape = [], tuple(in_shape)
        for L in self.layers:
            name, out, n = L.describe(shape)
            rows.append({"Tầng": name,
                         "Input shape": f"(batch, {', '.join(map(str, shape))})",
                         "Output shape": f"(batch, {', '.join(map(str, out))})",
                         "Số tham số": n,
                         "Huấn luyện được": "✓" if L.trainable else "—"})
            shape = out
        return rows

    def to_dict(self, decimals=6):
        head = {"binary": "sigmoid", "multiclass": "softmax", "regression": "linear"}[self.task]
        return {
            "kind": "cnn1d",
            "task": self.task,
            "head": head,
            "n_params": self.n_params(),
            "n_trainable_layers": sum(1 for L in self.layers if L.trainable),
            "layers": [L.to_dict(decimals) for L in self.layers],
        }


# ============================================================================
# 5. TIỆN ÍCH
# ============================================================================


def one_hot(y, n_classes):
    out = np.zeros((len(y), n_classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def forward_reference(bundle, x_single):
    """Bản tham chiếu của thuật toán suy luận sẽ viết lại bằng JavaScript.

    Nhận đúng dict đã ghi ra `model_cnn.json` (nên mọi trọng số đã bị làm tròn)
    và MỘT mẫu. Dùng trong notebook để kiểm tra parity NumPy ↔ JSON ↔ JS: nếu
    hàm này khớp với mô hình gốc thì bản JS chỉ cần dịch nguyên văn là đúng.

    x_single:
        - bài bảng  : mảng (C_in, L) hoặc (L,) — sẽ được thêm trục kênh
        - bài văn bản: mảng (L,) chỉ số token nguyên
    """
    a = np.asarray(x_single)
    layers = bundle["layers"]

    if layers[0]["type"] == "embedding":
        a = a.astype(np.int64)[None, :]                    # (1, L)
    else:
        if a.ndim == 1:
            a = a[None, :]                                 # (C_in=1, L)
        a = a.astype(float)[None, ...]                     # (1, C_in, L)

    for layer in layers:
        t = layer["type"]
        if t == "embedding":
            E = np.asarray(layer["E"])
            a = E[a].transpose(0, 2, 1)
        elif t == "conv1d":
            W = np.asarray(layer["W"])
            b = np.asarray(layer["b"])
            a, _ = conv1d_forward(a, W, b)
        elif t == "relu":
            a = relu(a)
        elif t == "maxpool1d":
            p = layer["p"]
            N, C, L = a.shape
            L_out = L // p
            a = a[:, :, :L_out * p].reshape(N, C, L_out, p).max(axis=3)
        elif t == "globalmaxpool1d":
            a = a.max(axis=2)
        elif t == "flatten":
            a = a.reshape(a.shape[0], -1)
        elif t == "dense":
            a = a @ np.asarray(layer["W"]) + np.asarray(layer["b"])
        elif t == "dropout":
            pass                                           # suy luận: dropout vô hiệu
        else:
            raise ValueError(f"Tầng lạ trong bundle: {t}")

    head = bundle["head"]
    if head == "sigmoid":
        return float(sigmoid(a).ravel()[0])
    if head == "softmax":
        return softmax(a.reshape(1, -1))[0]
    return float(a.ravel()[0])

### 4.1. Kiến trúc và bảng đặc tả tensor

**Định nghĩa 'Layer':** như Bài 1, chỉ các phép biến đổi **có tham số huấn
luyện được** (`Conv1D`, `Dense`) được đếm là một tầng. Kiến trúc dưới đây có
**5 tầng**. Điểm khác biệt so với Bài 1 nằm ở **head**:

$$\hat{z} = \text{Dense}_5(\cdot) \quad \text{(tuyến tính, không Sigmoid)}, \qquad
L = \frac{1}{n}\sum (\hat{z} - z)^2 \quad \text{(MSE)}$$

Đường đi của độ dài chuỗi: $19 \to 17 \to 15 \to 7 \to 5$.

In [8]:
# ============================================================================
# KHỐI 8 — Dựng CNN 5 tầng cho hồi quy (head tuyến tính + MSE)
# ============================================================================
HP = dict(lr=2e-3, l2=5e-3, dropout=0.15, batch_size=32, epochs=400, patience=50)
print("Siêu tham số (chọn trên tập VALIDATION theo R² thang log):")
print(HP, "\n")


def build_cnn(seed=SEED, dropout=HP["dropout"], lr=HP["lr"], l2=HP["l2"]):
    r = np.random.default_rng(seed)
    layers = [
        Conv1D(1, 16, 3, r), ReLU(),        # tầng 1: (1, 19)  -> (16, 17)
        Conv1D(16, 16, 3, r), ReLU(),       # tầng 2: (16, 17) -> (16, 15)
        MaxPool1D(2),                       #         (16, 15) -> (16, 7)
        Conv1D(16, 32, 3, r), ReLU(),       # tầng 3: (16, 7)  -> (32, 5)
        Flatten(),                          #         (32, 5)  -> (160,)
        Dropout(dropout, r),
        Dense(160, 16, r), ReLU(),          # tầng 4: 160 -> 16
        Dense(16, 1, r),                    # tầng 5: 16 -> 1 (TUYẾN TÍNH)
    ]
    return CNN1D(layers=layers, task="regression", lr=lr, l2=l2, seed=seed)


cnn = build_cnn()
shape_table = pd.DataFrame(cnn.shape_table((1, N_FEAT)))
shape_table.loc[len(shape_table)] = ["TỔNG", "", "", shape_table["Số tham số"].sum(), ""]
print(f"Tổng tham số: {cnn.n_params():,} — "
      f"{cnn.to_dict()['n_trainable_layers']} tầng huấn luyện được")
shape_table

Siêu tham số (chọn trên tập VALIDATION theo R² thang log):
{'lr': 0.002, 'l2': 0.005, 'dropout': 0.15, 'batch_size': 32, 'epochs': 400, 'patience': 50} 

Tổng tham số: 5,009 — 5 tầng huấn luyện được


,Tầng,Input shape,Output shape,Số tham số,Huấn luyện được
0,"Conv1D(C 1→16, K=3)","(batch, 1, 19)","(batch, 16, 17)",64,✓
1,ReLU,"(batch, 16, 17)","(batch, 16, 17)",0,—
2,"Conv1D(C 16→16, K=3)","(batch, 16, 17)","(batch, 16, 15)",784,✓
3,ReLU,"(batch, 16, 15)","(batch, 16, 15)",0,—
4,MaxPool1D(p=2),"(batch, 16, 15)","(batch, 16, 7)",0,—
5,"Conv1D(C 16→32, K=3)","(batch, 16, 7)","(batch, 32, 5)",1568,✓
6,ReLU,"(batch, 32, 5)","(batch, 32, 5)",0,—
7,Flatten,"(batch, 32, 5)","(batch, 160)",0,—
8,Dropout(p=0.15),"(batch, 160)","(batch, 160)",0,—
9,Dense(160→16),"(batch, 160)","(batch, 16)",2576,✓


### 4.2. Kiểm chứng gradient cho head hồi quy

Head tuyến tính + MSE có $\partial L/\partial z = 2(\hat{z} - z)/n$ — khác hệ số
so với head phân loại. Phải kiểm tra lại, không được giả định nó đúng chỉ vì
Bài 1 đã đúng.

In [9]:
# ============================================================================
# KHỐI 9 — Gradient check trên chính kiến trúc hồi quy sẽ dùng
# ============================================================================
def pool_relu_pattern(net):
    pat = []
    for L in net.layers:
        if isinstance(L, (MaxPool1D, GlobalMaxPool1D)):
            pat.append(L._argmax.copy())
        elif isinstance(L, ReLU):
            pat.append(L._z > 0)
    return pat


def same_pattern(a, b):
    return len(a) == len(b) and all(np.array_equal(u, v) for u, v in zip(a, b))


def gradient_check(net, X, yy, n_coord=25, h=1e-5):
    p, z = net.forward(X, training=False)
    base = pool_relu_pattern(net)
    net.backward(p, yy)
    grads = [g.copy() for L in net.layers for g in L.grad_list()]
    params = [q for L in net.layers for q in L.param_list()]
    worst, tested, skipped = 0.0, 0, 0
    for q, g in zip(params, grads):
        flat = q.ravel()
        for i in np.linspace(0, flat.size - 1, min(n_coord, flat.size)).astype(int):
            old = flat[i]
            flat[i] = old + h
            pp, zp = net.forward(X, training=False)
            lp, pat_p = net.loss(pp, yy, logits=zp), pool_relu_pattern(net)
            flat[i] = old - h
            pm, zm = net.forward(X, training=False)
            lm, pat_m = net.loss(pm, yy, logits=zm), pool_relu_pattern(net)
            flat[i] = old
            if not (same_pattern(base, pat_p) and same_pattern(base, pat_m)):
                skipped += 1
                continue
            num = (lp - lm) / (2 * h)
            worst = max(worst, abs(num - g.ravel()[i]) / max(1e-8, abs(num) + abs(g.ravel()[i])))
            tested += 1
    return worst, tested, skipped


worst, tested, skipped = gradient_check(build_cnn(seed=11), Xtr[:24], ztr[:24])
print(f"Sai số tương đối lớn nhất : {worst:.3e}")
print(f"Số toạ độ đã kiểm tra     : {tested}  (bỏ {skipped} toạ độ ở nếp gấp)")
assert worst < 1e-6, "Gradient check FAILED"
print("\n✅ Gradient của head hồi quy (Linear + MSE) là ĐÚNG.")

Sai số tương đối lớn nhất : 9.427e-07
Số toạ độ đã kiểm tra     : 133  (bỏ 57 toạ độ ở nếp gấp)

✅ Gradient của head hồi quy (Linear + MSE) là ĐÚNG.


## 5. Huấn luyện ba nền tảng

In [10]:
# ============================================================================
# KHỐI 10 — Huấn luyện bản NumPy from scratch
# ============================================================================
cnn = build_cnn()
t0 = time.time()
cnn.fit(Xtr, ztr, Xva, zva, epochs=HP["epochs"], batch_size=HP["batch_size"],
        patience=HP["patience"], verbose_every=25)
numpy_time = time.time() - t0
print(f"\n⏱ NumPy from scratch: {numpy_time:.2f}s "
      f"({len(cnn.history['train_loss'])} epoch)")

  epoch    1 | train_loss=0.3818 | train_metric=0.6185 | val_loss=0.3615 | val_metric=0.6248

  epoch   25 | train_loss=0.0643 | train_metric=0.9359 | val_loss=0.1023 | val_metric=0.8947


  epoch   50 | train_loss=0.0536 | train_metric=0.9467 | val_loss=0.0973 | val_metric=0.8999


  epoch   75 | train_loss=0.0510 | train_metric=0.9492 | val_loss=0.1012 | val_metric=0.8958


  ⏹ Early stopping tại epoch 95 (val_loss tốt nhất = 0.0883)

⏱ NumPy from scratch: 15.78s (95 epoch)


In [11]:
# ============================================================================
# KHỐI 11 — Bản PyTorch tương đương (channels-first, không cần transpose)
# ============================================================================
import torch
import torch.nn as nn

torch.manual_seed(SEED)

D = HP["dropout"]
torch_net = nn.Sequential(
    nn.Conv1d(1, 16, 3), nn.ReLU(),
    nn.Conv1d(16, 16, 3), nn.ReLU(),
    nn.MaxPool1d(2),
    nn.Conv1d(16, 32, 3), nn.ReLU(),
    nn.Flatten(),
    nn.Dropout(D),
    nn.Linear(160, 16), nn.ReLU(),
    nn.Linear(16, 1),
)
n_torch = sum(p.numel() for p in torch_net.parameters())
print(f"Tham số PyTorch = {n_torch:,} | NumPy = {cnn.n_params():,}")
assert n_torch == cnn.n_params()

Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
ztr_t = torch.tensor(ztr, dtype=torch.float32)
Xva_t = torch.tensor(Xva, dtype=torch.float32)
zva_t = torch.tensor(zva, dtype=torch.float32)
Xte_t = torch.tensor(Xte, dtype=torch.float32)

opt = torch.optim.Adam(torch_net.parameters(), lr=HP["lr"])
mse = nn.MSELoss()
torch_hist = {"train_loss": [], "val_loss": []}
g = torch.Generator().manual_seed(SEED)
best_vl, best_sd, wait = np.inf, None, 0

t0 = time.time()
for ep in range(1, HP["epochs"] + 1):
    torch_net.train()
    perm = torch.randperm(len(Xtr_t), generator=g)
    for s in range(0, len(perm), HP["batch_size"]):
        sl = perm[s:s + HP["batch_size"]]
        opt.zero_grad()
        l2p = sum((p ** 2).sum() for p in torch_net.parameters())
        loss = mse(torch_net(Xtr_t[sl]), ztr_t[sl]) + HP["l2"] * l2p / (2 * len(sl))
        loss.backward()
        opt.step()
    torch_net.eval()
    with torch.no_grad():
        tl = float(mse(torch_net(Xtr_t), ztr_t))
        vl = float(mse(torch_net(Xva_t), zva_t))
    torch_hist["train_loss"].append(tl)
    torch_hist["val_loss"].append(vl)
    if vl < best_vl - 1e-6:
        best_vl, wait = vl, 0
        best_sd = {k: v.clone() for k, v in torch_net.state_dict().items()}
    else:
        wait += 1
        if wait >= HP["patience"]:
            print(f"  ⏹ Early stopping tại epoch {ep} (val_loss = {best_vl:.4f})")
            break
    if ep % 25 == 0 or ep == 1:
        print(f"  epoch {ep:4d} | train_mse={tl:.4f} | val_mse={vl:.4f}")
if best_sd is not None:
    torch_net.load_state_dict(best_sd)
torch_time = time.time() - t0
print(f"\n⏱ PyTorch: {torch_time:.2f}s ({len(torch_hist['train_loss'])} epoch)")

Tham số PyTorch = 5,009 | NumPy = 5,009


  epoch    1 | train_mse=0.8137 | val_mse=0.8004


  epoch   25 | train_mse=0.0686 | val_mse=0.0828


  epoch   50 | train_mse=0.0759 | val_mse=0.0938


  epoch   75 | train_mse=0.0582 | val_mse=0.0862


  ⏹ Early stopping tại epoch 81 (val_loss = 0.0786)

⏱ PyTorch: 17.20s (81 epoch)


In [12]:
# ============================================================================
# KHỐI 12 — Bản TensorFlow/Keras (channels-last: phải transpose sang (N, L, C))
# ============================================================================
import tensorflow as tf
from tensorflow import keras

tf.keras.utils.set_random_seed(SEED)

Xtr_tf = np.transpose(Xtr, (0, 2, 1))
Xva_tf = np.transpose(Xva, (0, 2, 1))
Xte_tf = np.transpose(Xte, (0, 2, 1))
print("channels-first:", Xtr.shape, " -> channels-last:", Xtr_tf.shape)

reg = keras.regularizers.l2(HP["l2"] / 2)
tf_net = keras.Sequential([
    keras.layers.Input(shape=(N_FEAT, 1)),
    keras.layers.Conv1D(16, 3, activation="relu", kernel_regularizer=reg),
    keras.layers.Conv1D(16, 3, activation="relu", kernel_regularizer=reg),
    keras.layers.MaxPooling1D(2),
    keras.layers.Conv1D(32, 3, activation="relu", kernel_regularizer=reg),
    keras.layers.Flatten(),
    keras.layers.Dropout(D),
    keras.layers.Dense(16, activation="relu", kernel_regularizer=reg),
    keras.layers.Dense(1, activation=None),        # head TUYẾN TÍNH
])
assert tf_net.count_params() == cnn.n_params()
print(f"Tham số TensorFlow = {tf_net.count_params():,}  ✅ khớp hai bản kia")

tf_net.compile(optimizer=keras.optimizers.Adam(learning_rate=HP["lr"]), loss="mse")
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=HP["patience"],
                                   restore_best_weights=True, verbose=0)
t0 = time.time()
tf_hist = tf_net.fit(Xtr_tf, ztr, validation_data=(Xva_tf, zva),
                     epochs=HP["epochs"], batch_size=HP["batch_size"],
                     callbacks=[es], verbose=0)
tf_time = time.time() - t0
print(f"⏱ TensorFlow: {tf_time:.2f}s ({len(tf_hist.history['loss'])} epoch)")

channels-first: (1399, 1, 19)  -> channels-last: (1399, 19, 1)
Tham số TensorFlow = 5,009  ✅ khớp hai bản kia


⏱ TensorFlow: 56.08s (229 epoch)


## 6. Các mô hình đối sánh

In [13]:
# ============================================================================
# KHỐI 13 — MLP 5 tầng (A03) + ba mô hình Học máy truyền thống
# ============================================================================
torch.manual_seed(SEED)
mlp_net = nn.Sequential(
    nn.Linear(N_FEAT, 128), nn.ReLU(), nn.Dropout(0.15),
    nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.15),
    nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.15),
    nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.15),
    nn.Linear(16, 1),
)
Xtr_m = torch.tensor(Xtr_flat, dtype=torch.float32)
Xva_m = torch.tensor(Xva_flat, dtype=torch.float32)
Xte_m = torch.tensor(Xte_flat, dtype=torch.float32)
opt_m = torch.optim.Adam(mlp_net.parameters(), lr=1e-3)
gm = torch.Generator().manual_seed(SEED)
best_vl_m, best_sd_m, wait_m = np.inf, None, 0
for ep in range(1, 401):
    mlp_net.train()
    perm = torch.randperm(len(Xtr_m), generator=gm)
    for s in range(0, len(perm), 32):
        sl = perm[s:s + 32]
        opt_m.zero_grad()
        loss = mse(mlp_net(Xtr_m[sl]), ztr_t[sl])
        loss.backward()
        opt_m.step()
    mlp_net.eval()
    with torch.no_grad():
        vl = float(mse(mlp_net(Xva_m), zva_t))
    if vl < best_vl_m - 1e-6:
        best_vl_m, wait_m = vl, 0
        best_sd_m = {k: v.clone() for k, v in mlp_net.state_dict().items()}
    else:
        wait_m += 1
        if wait_m >= 50:
            break
if best_sd_m is not None:
    mlp_net.load_state_dict(best_sd_m)
print(f"MLP-5 (A03): {sum(p.numel() for p in mlp_net.parameters()):,} tham số, "
      f"dừng epoch {ep}, val_mse = {best_vl_m:.4f}")

classical = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=250, max_depth=14,
                                           random_state=SEED, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, max_depth=4,
                                                   learning_rate=0.06,
                                                   random_state=SEED),
}
for name, mdl in classical.items():
    mdl.fit(Xtr_flat, ztr.ravel())
    print(f"  {name:<20} val R²(log) = {r2_score(zva, mdl.predict(Xva_flat)):.4f}")

MLP-5 (A03): 13,441 tham số, dừng epoch 62, val_mse = 0.0773
  Linear Regression    val R²(log) = 0.9072


  Random Forest        val R²(log) = 0.9033


  Gradient Boosting    val R²(log) = 0.9115


## 7. Đánh giá trên tập kiểm thử

Mô hình học trên thang `z = (log1p(Price) - mean) / std`, nên để báo cáo số liệu
có ý nghĩa kinh tế ta phải **nghịch đảo phép biến đổi** về tỉ VNĐ:

$$\widehat{\text{Price}} = \exp\!\big(\hat{z}\cdot \sigma + \mu\big) - 1$$

Ta báo cáo cả hai thang: $R^2$ trên thang log (thang mà mô hình tối ưu) và
MAE/RMSE trên thang gốc (thang mà người dùng quan tâm).

In [14]:
# ============================================================================
# KHỐI 14 — Dự đoán của mọi mô hình, quy về cả hai thang
# ============================================================================
def to_price(z):
    """Nghịch đảo (log1p + chuẩn hoá) để về lại tỉ VNĐ."""
    return np.expm1(np.asarray(z).ravel() * Y_STD + Y_MEAN)


pred_z = {}
pred_z["CNN-5 (NumPy)"] = cnn.predict_proba(Xte).ravel()
torch_net.eval()
with torch.no_grad():
    pred_z["CNN-5 (PyTorch)"] = torch_net(Xte_t).numpy().ravel()
    pred_z["MLP-5 (A03, PyTorch)"] = mlp_net(Xte_m).numpy().ravel()
pred_z["CNN-5 (TensorFlow)"] = tf_net.predict(Xte_tf, verbose=0).ravel()
for name, mdl in classical.items():
    pred_z[name] = mdl.predict(Xte_flat).ravel()

price_true = y_test.ravel()
z_true = zte.ravel()


def evaluate(pz):
    price_pred = to_price(pz)
    return {
        "R² (log)": r2_score(z_true, pz),
        "MAE (tỉ)": mean_absolute_error(price_true, price_pred),
        "RMSE (tỉ)": float(np.sqrt(mean_squared_error(price_true, price_pred))),
        "MAPE (%)": float(np.mean(np.abs((price_pred - price_true) / price_true)) * 100),
        "R² (gốc)": r2_score(price_true, price_pred),
    }


res_df = pd.DataFrame({k: evaluate(v) for k, v in pred_z.items()}).T
res_df = res_df.sort_values("R² (log)", ascending=False)
print("Kết quả trên tập TEST:\n")
res_df.round(4)

Kết quả trên tập TEST:



,R² (log),MAE (tỉ),RMSE (tỉ),MAPE (%),R² (gốc)
"MLP-5 (A03, PyTorch)",0.9333,3.5803,6.4451,16.9278,0.8774
Gradient Boosting,0.9300,3.5487,6.2588,17.2771,0.8844
CNN-5 (TensorFlow),0.9283,3.5616,6.2376,17.6402,0.8852
CNN-5 (PyTorch),0.9278,3.6504,6.5029,17.6549,0.8752
CNN-5 (NumPy),0.9254,3.7478,6.6864,17.6278,0.8680
Random Forest,0.9246,3.6013,6.5715,17.0097,0.8725
Linear Regression,0.9148,4.2037,10.6988,18.9598,0.6621


In [15]:
# ============================================================================
# KHỐI 15 — Độ lệch giữa ba nền tảng trên cùng tập test
# ============================================================================
tri = ["CNN-5 (NumPy)", "CNN-5 (PyTorch)", "CNN-5 (TensorFlow)"]
rows = []
for a, b in [(tri[0], tri[1]), (tri[0], tri[2]), (tri[1], tri[2])]:
    pa, pb = to_price(pred_z[a]), to_price(pred_z[b])
    rows.append({
        "Cặp so sánh": f"{a.split('(')[1][:-1]} ↔ {b.split('(')[1][:-1]}",
        "Sai khác TB (tỉ VNĐ)": float(np.mean(np.abs(pa - pb))),
        "Sai khác lớn nhất (tỉ)": float(np.abs(pa - pb).max()),
        "Tương quan Pearson": float(np.corrcoef(pa, pb)[0, 1]),
    })
divergence = pd.DataFrame(rows)
print(f"Tập test có {len(price_true)} bất động sản.\n")
divergence.round(6)

Tập test có 300 bất động sản.



,Cặp so sánh,Sai khác TB (tỉ VNĐ),Sai khác lớn nhất (tỉ),Tương quan Pearson
0,NumPy ↔ PyTorch,1.976286,20.862223,0.979483
1,NumPy ↔ TensorFlow,1.540748,9.381218,0.989495
2,PyTorch ↔ TensorFlow,1.631827,21.164474,0.987608


## 8. Trực quan hoá

In [16]:
# ============================================================================
# KHỐI 16 — Hình 3: đường cong học của ba nền tảng
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, (tr, va), title in [
    (axes[0], (cnn.history["train_loss"], cnn.history["val_loss"]), "(a) NumPy from scratch"),
    (axes[1], (torch_hist["train_loss"], torch_hist["val_loss"]), "(b) PyTorch"),
    (axes[2], (tf_hist.history["loss"], tf_hist.history["val_loss"]), "(c) TensorFlow/Keras"),
]:
    ax.plot(tr, label="Train", color="#2563eb", linewidth=1.8)
    ax.plot(va, label="Validation", color="#ef4444", linewidth=1.8)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylim(0, max(max(tr[:5]), max(va[:5])) * 1.05)
    ax.legend()
axes[0].set_ylabel("MSE (thang log chuẩn hoá)")
plt.suptitle("Hình 3 — Đường cong học của cùng một CNN hồi quy trên ba nền tảng",
             fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(FIG / "c2_fig3_curves.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_17640\1958892572.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# ============================================================================
# KHỐI 17 — Hình 4: (a) dự đoán vs thực tế, (b) phân bố sai số, (c) sai số theo nhóm giá
# ============================================================================
best_cnn_pred = to_price(pred_z["CNN-5 (NumPy)"])

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8))

axes[0].scatter(price_true, best_cnn_pred, s=16, alpha=0.5, color="#2563eb")
lim = [min(price_true.min(), best_cnn_pred.min()) * 0.8,
       max(price_true.max(), best_cnn_pred.max()) * 1.2]
axes[0].plot(lim, lim, "r--", linewidth=1.4, label="Dự đoán hoàn hảo")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlim(lim)
axes[0].set_ylim(lim)
axes[0].set_xlabel("Giá thực tế (tỉ VNĐ, log)")
axes[0].set_ylabel("Giá dự đoán (tỉ VNĐ, log)")
axes[0].set_title(f"(a) Dự đoán vs Thực tế — R²(log) = "
                  f"{res_df.loc['CNN-5 (NumPy)', 'R² (log)']:.4f}", fontweight="bold")
axes[0].legend()

resid = np.log1p(best_cnn_pred) - np.log1p(price_true)
axes[1].hist(resid, bins=45, color="#10b981", edgecolor="white")
axes[1].axvline(0, color="#111827", ls="--", linewidth=1.3)
axes[1].set_title(f"(b) Phân bố phần dư trên thang log\n"
                  f"(TB = {resid.mean():+.4f}, độ lệch chuẩn = {resid.std():.4f})",
                  fontweight="bold", fontsize=9.5)
axes[1].set_xlabel("log1p(dự đoán) − log1p(thực tế)")
axes[1].set_ylabel("Số bất động sản")

q = pd.qcut(price_true, 4, labels=["Rẻ nhất 25%", "25–50%", "50–75%", "Đắt nhất 25%"])
grp = pd.DataFrame({"q": q, "ape": np.abs((best_cnn_pred - price_true) / price_true) * 100})
med = grp.groupby("q", observed=True)["ape"].median()
axes[2].bar(range(len(med)), med.values, color="#f59e0b", width=0.62)
for i, v in enumerate(med.values):
    axes[2].text(i, v + 0.7, f"{v:.1f}%", ha="center", fontweight="bold", fontsize=9)
axes[2].set_xticks(range(len(med)))
axes[2].set_xticklabels(med.index, rotation=18, ha="right", fontsize=8)
axes[2].set_title("(c) Sai số phần trăm trung vị theo nhóm giá", fontweight="bold")
axes[2].set_ylabel("|Sai số| / Giá thực tế (%)")
axes[2].set_ylim(0, med.max() * 1.28)

plt.suptitle("Hình 4 — Chất lượng dự đoán của CNN hồi quy", fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(FIG / "c2_fig4_error.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_17640\4069644477.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# ============================================================================
# KHỐI 18 — Hình 5: đối sánh mô hình và chi phí huấn luyện
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

order_plot = res_df.sort_values("R² (log)").index
colors = ["#2563eb" if "CNN" in n else "#a855f7" if "MLP" in n else "#94a3b8"
          for n in order_plot]
vals = res_df.loc[order_plot, "R² (log)"].values
axes[0].barh(range(len(order_plot)), vals, color=colors)
axes[0].set_yticks(range(len(order_plot)))
axes[0].set_yticklabels(order_plot, fontsize=8)
for i, v in enumerate(vals):
    axes[0].text(v + 0.004, i, f"{v:.4f}", va="center", fontsize=8, fontweight="bold")
axes[0].set_xlim(min(vals) - 0.05, max(vals) + 0.04)
axes[0].set_title("(a) R² trên thang log — tập test", fontweight="bold")
axes[0].set_xlabel("R² (log)")

names_t = ["NumPy\nfrom scratch", "PyTorch", "TensorFlow"]
times = [numpy_time, torch_time, tf_time]
epochs_run = [len(cnn.history["train_loss"]), len(torch_hist["train_loss"]),
              len(tf_hist.history["loss"])]
per_ep = [t / e for t, e in zip(times, epochs_run)]
bars = axes[1].bar(names_t, per_ep, color=["#2563eb", "#16a34a", "#f59e0b"], width=0.55)
for bar, t, e in zip(bars, times, epochs_run):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                 f"{bar.get_height()*1000:.0f} ms/epoch\n({t:.1f}s / {e} ep)",
                 ha="center", fontsize=8, fontweight="bold")
axes[1].set_title("(b) Thời gian huấn luyện trên mỗi epoch", fontweight="bold")
axes[1].set_ylabel("giây / epoch")
axes[1].set_ylim(0, max(per_ep) * 1.32)

plt.suptitle("Hình 5 — Đối sánh mô hình và chi phí huấn luyện",
             fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(FIG / "c2_fig5_compare.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_17640\1706131498.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Xuất mô hình cho web app

In [19]:
# ============================================================================
# KHỐI 19 — Đóng gói model_cnn.json
# Bundle chứa cả tham số nghịch đảo biến đổi nhãn (Y_MEAN, Y_STD) để frontend
# quy đổi dự đoán về tỉ VNĐ.
# ============================================================================
bundle = cnn.to_dict(decimals=6)
bundle.update({
    "model_name": "CNN 1D 5 tầng — hồi quy (NumPy from scratch)",
    "assignment": "Assignment 04",
    "input_shape": [1, N_FEAT],
    "feature_names": FEATURE_NAMES,
    "numeric_features": NUM,
    "provinces": provinces,
    "house_types": house_types,
    "scaler": {"mean_": scaler.mean_.tolist(), "scale_": scaler.scale_.tolist()},
    "target_transform": {"type": "log1p_zscore", "mean": Y_MEAN, "std": Y_STD},
    "metrics": {
        "r2_log": float(res_df.loc["CNN-5 (NumPy)", "R² (log)"]),
        "mae": float(res_df.loc["CNN-5 (NumPy)", "MAE (tỉ)"]),
        "rmse": float(res_df.loc["CNN-5 (NumPy)", "RMSE (tỉ)"]),
        "mape": float(res_df.loc["CNN-5 (NumPy)", "MAPE (%)"]),
        "r2_raw": float(res_df.loc["CNN-5 (NumPy)", "R² (gốc)"]),
    },
    "comparison": {
        name: {"r2_log": float(res_df.loc[name, "R² (log)"]),
               "mae": float(res_df.loc[name, "MAE (tỉ)"]),
               "rmse": float(res_df.loc[name, "RMSE (tỉ)"]),
               "mape": float(res_df.loc[name, "MAPE (%)"])}
        for name in res_df.index
    },
    "framework_divergence": divergence.to_dict(orient="records"),
    "architecture_text": f"{N_FEAT} → Conv(16) → Conv(16) → Pool → Conv(32) → Dense(16) → 1 (Linear)",
    "training": {
        "epochs_run": len(cnn.history["train_loss"]),
        "batch_size": HP["batch_size"], "lr": HP["lr"],
        "l2": HP["l2"], "dropout": HP["dropout"],
        "numpy_seconds": round(numpy_time, 2),
        "pytorch_seconds": round(torch_time, 2),
        "tensorflow_seconds": round(tf_time, 2),
    },
    "history": {
        "train_loss": [round(v, 5) for v in cnn.history["train_loss"]],
        "val_loss": [round(v, 5) for v in cnn.history["val_loss"]],
        "train_r2": [round(v, 5) for v in cnn.history["train_metric"]],
        "val_r2": [round(v, 5) for v in cnn.history["val_metric"]],
    },
})

out_path = ROOT / "model_cnn.json"
out_path.write_text(json.dumps(bundle, ensure_ascii=False, separators=(",", ":")),
                    encoding="utf-8")
print(f"✅ Đã ghi {out_path.name} — {out_path.stat().st_size/1024:.1f} KB")
print(f"   {bundle['n_params']:,} tham số, {bundle['n_trainable_layers']} tầng huấn luyện được")

✅ Đã ghi model_cnn.json — 54.0 KB
   5,009 tham số, 5 tầng huấn luyện được


In [20]:
# ============================================================================
# KHỐI 20 — Parity NumPy <-> JSON trên toàn bộ tập test (thang tỉ VNĐ)
# ============================================================================
reloaded = json.loads(out_path.read_text(encoding="utf-8"))
max_err_z, max_err_price = 0.0, 0.0
for i in range(len(Xte)):
    ref_z = forward_reference(reloaded, Xte[i])
    max_err_z = max(max_err_z, abs(ref_z - float(pred_z["CNN-5 (NumPy)"][i])))
    max_err_price = max(max_err_price,
                        abs(float(to_price([ref_z])[0]) - float(best_cnn_pred[i])))
print(f"Sai số lớn nhất trên thang z      : {max_err_z:.3e}")
print(f"Sai số lớn nhất trên thang tỉ VNĐ : {max_err_price:.3e}")
assert max_err_z < 1e-5, "Parity FAILED"
print("✅ Parity PASSED — web app sẽ cho kết quả trùng khớp với notebook.")

Sai số lớn nhất trên thang z      : 5.521e-06


Sai số lớn nhất trên thang tỉ VNĐ : 1.682e-04
✅ Parity PASSED — web app sẽ cho kết quả trùng khớp với notebook.


In [21]:
# ============================================================================
# KHỐI 21 — Mẫu tham chiếu cho kiểm thử JavaScript
# ============================================================================
samples = []
for i in range(min(12, len(Xte))):
    samples.append({
        "scaled": [round(float(v), 8) for v in Xte[i, 0]],
        "expected_z": round(float(pred_z["CNN-5 (NumPy)"][i]), 10),
        "expected_price": round(float(best_cnn_pred[i]), 8),
        "actual_price": float(price_true[i]),
    })
samp_path = ROOT / "model_cnn_samples.json"
samp_path.write_text(json.dumps(samples, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"✅ Đã ghi {samp_path.name} — {len(samples)} mẫu tham chiếu")

✅ Đã ghi model_cnn_samples.json — 12 mẫu tham chiếu


## 10. Kết luận Bài toán 2

In [22]:
# ============================================================================
# KHỐI 22 — Tóm tắt số liệu chính
# ============================================================================
cnn_r2 = res_df.loc["CNN-5 (NumPy)", "R² (log)"]
mlp_r2 = res_df.loc["MLP-5 (A03, PyTorch)", "R² (log)"]
best_classical = max(classical, key=lambda k: res_df.loc[k, "R² (log)"])
bc_r2 = res_df.loc[best_classical, "R² (log)"]

print("=" * 70)
print("BÀI TOÁN 2 — DỰ ĐOÁN GIÁ NHÀ BẰNG CNN 1D (HỒI QUY)")
print("=" * 70)
print(f"Kiến trúc      : {bundle['architecture_text']}")
print(f"Tham số        : {cnn.n_params():,} ({bundle['n_trainable_layers']} tầng huấn luyện được)")
print(f"Gradient check : sai số tương đối {worst:.2e} — head Linear + MSE ĐÚNG")
print()
for n in ["CNN-5 (NumPy)", "CNN-5 (PyTorch)", "CNN-5 (TensorFlow)",
          "MLP-5 (A03, PyTorch)", best_classical]:
    print(f"{n:<22}: R²(log) = {res_df.loc[n, 'R² (log)']:.4f}  |  "
          f"MAE = {res_df.loc[n, 'MAE (tỉ)']:.3f} tỉ  |  "
          f"MAPE = {res_df.loc[n, 'MAPE (%)']:.1f}%")
print()
print(f"CNN so với MLP-5          : {cnn_r2 - mlp_r2:+.4f}")
print(f"CNN so với {best_classical:<14}: {cnn_r2 - bc_r2:+.4f}")
print()
print("Thời gian huấn luyện (giây/epoch):")
for nm, t, e in zip(["NumPy", "PyTorch", "TensorFlow"], times, epochs_run):
    print(f"  {nm:<12}: {t/e*1000:7.1f} ms/epoch  (tổng {t:.1f}s, {e} epoch)")
print()
print(f"Cấu trúc khối của chuỗi đầu vào: {len(NUM)} cột số + "
      f"{len(provinces)} + {len(house_types)} cột one-hot")
print(f"  {n_pure_onehot}/{len(win_onehot)} cửa sổ tích chập nằm TRỌN trong khối one-hot")
print("  -> ở những cửa sổ đó, tích chập chỉ là một bảng tra cứu.")
print("=" * 70)

BÀI TOÁN 2 — DỰ ĐOÁN GIÁ NHÀ BẰNG CNN 1D (HỒI QUY)
Kiến trúc      : 19 → Conv(16) → Conv(16) → Pool → Conv(32) → Dense(16) → 1 (Linear)
Tham số        : 5,009 (5 tầng huấn luyện được)
Gradient check : sai số tương đối 9.43e-07 — head Linear + MSE ĐÚNG

CNN-5 (NumPy)         : R²(log) = 0.9254  |  MAE = 3.748 tỉ  |  MAPE = 17.6%
CNN-5 (PyTorch)       : R²(log) = 0.9278  |  MAE = 3.650 tỉ  |  MAPE = 17.7%
CNN-5 (TensorFlow)    : R²(log) = 0.9283  |  MAE = 3.562 tỉ  |  MAPE = 17.6%
MLP-5 (A03, PyTorch)  : R²(log) = 0.9333  |  MAE = 3.580 tỉ  |  MAPE = 16.9%
Gradient Boosting     : R²(log) = 0.9300  |  MAE = 3.549 tỉ  |  MAPE = 17.3%

CNN so với MLP-5          : -0.0079
CNN so với Gradient Boosting: -0.0046

Thời gian huấn luyện (giây/epoch):
  NumPy       :   166.1 ms/epoch  (tổng 15.8s, 95 epoch)
  PyTorch     :   212.3 ms/epoch  (tổng 17.2s, 81 epoch)
  TensorFlow  :   244.9 ms/epoch  (tổng 56.1s, 229 epoch)

Cấu trúc khối của chuỗi đầu vào: 4 cột số + 10 + 5 cột one-hot
  13/17 cửa sổ 

### Nhận xét trung thực

1. **Head phải theo bài toán.** Chuyển từ phân loại sang hồi quy chỉ cần đổi
   nơ-ron ra sang tuyến tính và đổi loss sang MSE — nhưng đó là thay đổi **bắt
   buộc**: giữ Sigmoid sẽ chặn dự đoán trong $[0,1]$ và mô hình không thể học.
   Gradient check xác nhận hệ số $2(\hat z - z)/n$ của head mới là đúng.

2. **Biến đổi nhãn quan trọng hơn kiến trúc.** Việc học trên `log1p(Price)`
   quyết định chất lượng mô hình nhiều hơn mọi lựa chọn về số bộ lọc hay độ sâu.
   Nó biến bài toán "sai số tuyệt đối" thành "sai số tương đối", đúng với cách
   thị trường bất động sản vận hành.

3. **Cửa sổ tích chập trên khối one-hot gần như vô nghĩa.** Đây là điểm yếu mô
   hình hoá nặng hơn cả Bài 1: phần lớn cửa sổ nằm trọn trong khối chỉ báo, nơi
   nhiều nhất một giá trị bằng 1. Tích chập khi đó suy biến thành phép tra bảng,
   và việc tỉnh nào bị gộp chung cửa sổ hoàn toàn do **thứ tự bảng chữ cái**.
   Không có bất kỳ cơ sở thực tế nào cho cách nhóm đó.

4. **Kết quả phản ánh đúng điều trên.** CNN không vượt được Gradient Boosting —
   mô hình vốn xử lý biến hạng mục và quan hệ phi tuyến trên dữ liệu bảng rất
   tốt. Đây là một kết quả **âm tính có giá trị**: nó cho thấy chọn kiến trúc
   phải xuất phát từ **cấu trúc thật của dữ liệu**, không phải từ độ "hiện đại"
   của mô hình (slide 28: *Neural network architecture = Data + Task +
   Representation*).